# NER Framework Evaluation — Limbus Company Mechanic Extraction

Compares **spaCy custom NER** (trained on annotated samples) vs. **rule-based extraction** (regex + spaCy PhraseMatcher) for tagging game mechanics in identity text.

**Test data:** `docs/parsed-ids/*.md` (3 identities with different mechanic profiles)

**Target entities:**
- Status effects: Bleed, Burn, Tremor, Rupture, Sinking, Poise, Charge
- Unique mechanics: Corpus Ingredient, Iron Maiden, Artwork: Fascia, etc.
- Conditional triggers: `[On Hit]`, `[On Use]`, `[Clash Win]`, `at X+ count`
- Stat modifiers: Defense Level Up/Down, Offense Level Up/Down, Bind, Haste

In [ ]:
# pip install spacy
# python -m spacy download en_core_web_sm

import re
import glob
from pathlib import Path
from collections import Counter

import spacy
from spacy.matcher import PhraseMatcher, Matcher
from spacy.tokens import Span

## 1. Load parsed identity markdown files

In [ ]:
PARSED_IDS_DIR = Path("../docs/parsed-ids")

identity_texts = {}
for md_file in sorted(PARSED_IDS_DIR.glob("*.md")):
    identity_texts[md_file.stem] = md_file.read_text(encoding="utf-8")

print(f"Loaded {len(identity_texts)} identities: {list(identity_texts.keys())}")

## 2. Define the mechanic vocabulary (gold standard)

Controlled vocabulary of game mechanics, used both as the rule-based dictionary and as the gold-standard annotation set.

In [ ]:
STATUS_EFFECTS = [
    "Bleed", "Burn", "Tremor", "Rupture", "Sinking", "Poise", "Charge",
    "Bind", "Haste", "Protection", "Shield",
]

STAT_MODIFIERS = [
    "Defense Level Up", "Defense Level Down",
    "Offense Level Up", "Offense Level Down",
    "Damage Up", "Damage Down",
    "Slash DMG Up", "Pierce DMG Up", "Blunt DMG Up",
    "Clash Power", "Coin Power", "Base Power", "Final Power",
    "Atk Weight",
]

UNIQUE_MECHANICS = [
    "Iron Maiden", "Corpus Ingredient", "Artwork: Fascia",
    "The Self Unbound", "Flow State", "Assist Defense",
    "Somatic Frisson-inspiring Melody",
    "Unbreakable Coin",
]

TRIGGERS = [
    "[On Hit]", "[On Use]", "[Clash Win]", "[Heads Hit]",
    "[On Evade]", "[Combat Start]", "[Attack End]",
    "[Skill End]",
]

# Flattened list for evaluation
ALL_MECHANICS = STATUS_EFFECTS + STAT_MODIFIERS + UNIQUE_MECHANICS
ALL_TRIGGERS = TRIGGERS

print(f"Vocabulary: {len(ALL_MECHANICS)} mechanics, {len(ALL_TRIGGERS)} triggers")

## 3. Approach A — Rule-based extraction (regex + PhraseMatcher)

In [ ]:
nlp = spacy.load("en_core_web_sm")

# --- PhraseMatcher for exact mechanic terms ---
phrase_matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
mechanic_patterns = [nlp.make_doc(term) for term in ALL_MECHANICS]
phrase_matcher.add("GAME_MECHANIC", mechanic_patterns)

# --- Regex patterns for triggers and conditional expressions ---
TRIGGER_RE = re.compile(r"\[(On Hit|On Use|Clash Win|Heads Hit|On Evade|Combat Start|Attack End|Skill End)\]")
CONDITIONAL_RE = re.compile(r"(?:at|if|for every)\s+(\d+)\+?\s+([A-Z][a-z]+(?:\s+[A-Z]?[a-z]+)*)", re.IGNORECASE)
COUNT_POTENCY_RE = re.compile(r"(\+?\d+)\s+(Bleed|Burn|Tremor|Rupture|Sinking|Poise|Charge|Bind|Haste)\s*(Count|Potency)?", re.IGNORECASE)


def extract_rule_based(text: str) -> dict:
    """Extract mechanics using PhraseMatcher + regex."""
    doc = nlp(text)

    # PhraseMatcher hits
    matches = phrase_matcher(doc)
    mechanic_hits = []
    for match_id, start, end in matches:
        span_text = doc[start:end].text
        mechanic_hits.append(span_text)

    # Regex: triggers
    trigger_hits = TRIGGER_RE.findall(text)

    # Regex: conditionals ("at 10+ Corpus Ingredient Count")
    conditional_hits = CONDITIONAL_RE.findall(text)

    # Regex: count/potency inflictions
    count_potency_hits = COUNT_POTENCY_RE.findall(text)

    return {
        "mechanics": Counter(mechanic_hits),
        "triggers": Counter(trigger_hits),
        "conditionals": conditional_hits,
        "count_potency": count_potency_hits,
    }


# Run on all identities
rule_results = {}
for name, text in identity_texts.items():
    rule_results[name] = extract_rule_based(text)
    print(f"\n=== {name} ===")
    print(f"  Mechanics found: {dict(rule_results[name]['mechanics'])}")
    print(f"  Triggers found:  {dict(rule_results[name]['triggers'])}")
    print(f"  Conditionals:    {rule_results[name]['conditionals'][:5]}")

## 4. Approach B — spaCy custom NER (EntityRuler as proxy for trained NER)

For a PoC, we use spaCy's `EntityRuler` (pattern-based NER component) to simulate what a trained NER model would do. A fully trained model would require annotated training data (50-100+ examples); EntityRuler lets us evaluate the entity schema quickly.

In [ ]:
nlp_ner = spacy.load("en_core_web_sm")

ruler = nlp_ner.add_pipe("entity_ruler", before="ner")

patterns = []
for effect in STATUS_EFFECTS:
    patterns.append({"label": "STATUS_EFFECT", "pattern": effect})
    # Also match "X Count" and "X Potency" variants
    patterns.append({"label": "STATUS_EFFECT", "pattern": f"{effect} Count"})
    patterns.append({"label": "STATUS_EFFECT", "pattern": f"{effect} Potency"})

for mod in STAT_MODIFIERS:
    patterns.append({"label": "STAT_MODIFIER", "pattern": mod})

for mech in UNIQUE_MECHANICS:
    patterns.append({"label": "UNIQUE_MECHANIC", "pattern": mech})

ruler.add_patterns(patterns)

# Run on all identities
ner_results = {}
for name, text in identity_texts.items():
    doc = nlp_ner(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    ner_results[name] = Counter(entities)
    print(f"\n=== {name} ===")
    for (ent_text, ent_label), count in ner_results[name].most_common(15):
        print(f"  {ent_label:20s} | {ent_text:30s} | x{count}")

## 5. Build gold-standard annotation for evaluation

Manually defined expected mechanics per identity, used to compute precision/recall.

In [ ]:
GOLD_STANDARD = {
    "Ring_Apprentice_Faust": {
        "primary_mechanics": ["Bleed", "Corpus Ingredient", "Iron Maiden"],
        "secondary_mechanics": ["Bind", "Haste", "Protection", "Shield"],
        "unique_mechanics": ["Iron Maiden", "Corpus Ingredient", "Artwork: Fascia",
                             "The Self Unbound", "Flow State", "Assist Defense",
                             "Unbreakable Coin"],
        "stat_modifiers": ["Defense Level Up", "Defense Level Down", "Damage Down",
                           "Slash DMG Up", "Clash Power", "Final Power", "Base Power",
                           "Coin Power", "Atk Weight"],
    },
    "Blade_Lineage_Salsu_Yi_Sang": {
        "primary_mechanics": ["Poise"],
        "secondary_mechanics": [],
        "unique_mechanics": [],
        "stat_modifiers": ["Coin Power"],
    },
    "Ring_Pointillist_Student_Yi_Sang": {
        "primary_mechanics": ["Bleed"],
        "secondary_mechanics": ["Burn", "Tremor", "Rupture", "Sinking"],
        "unique_mechanics": [],
        "stat_modifiers": ["Clash Power", "Coin Power", "Base Power",
                           "Offense Level Up"],
    },
}


def evaluate_extraction(extracted_mechanics: set, gold: dict) -> dict:
    """Compute precision, recall, F1 against gold-standard mechanics."""
    gold_set = set()
    for category in gold.values():
        gold_set.update(category)

    tp = extracted_mechanics & gold_set
    fp = extracted_mechanics - gold_set
    fn = gold_set - extracted_mechanics

    precision = len(tp) / (len(tp) + len(fp)) if (len(tp) + len(fp)) > 0 else 0
    recall = len(tp) / (len(tp) + len(fn)) if (len(tp) + len(fn)) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {"precision": precision, "recall": recall, "f1": f1,
            "tp": tp, "fp": fp, "fn": fn}

## 6. Compare results

In [ ]:
print("=" * 80)
print("RULE-BASED vs ENTITY-RULER NER — Evaluation against gold standard")
print("=" * 80)

for identity_name, gold in GOLD_STANDARD.items():
    print(f"\n--- {identity_name} ---")

    # Rule-based: extract unique mechanic names from the Counter
    rb_mechanics = set(rule_results[identity_name]["mechanics"].keys())
    rb_eval = evaluate_extraction(rb_mechanics, gold)
    print(f"  Rule-based:  P={rb_eval['precision']:.2f}  R={rb_eval['recall']:.2f}  F1={rb_eval['f1']:.2f}")
    if rb_eval["fn"]:
        print(f"    Missed: {rb_eval['fn']}")
    if rb_eval["fp"]:
        print(f"    Extra:  {rb_eval['fp']}")

    # EntityRuler NER: extract unique entity texts
    ner_mechanics = set()
    for (ent_text, _label), _count in ner_results[identity_name].items():
        # Normalize: strip " Count" / " Potency" suffixes for matching
        base = ent_text.replace(" Count", "").replace(" Potency", "")
        ner_mechanics.add(base)
    ner_mechanics.update(ent_text for (ent_text, _), _ in ner_results[identity_name].items())
    ner_eval = evaluate_extraction(ner_mechanics, gold)
    print(f"  EntityRuler: P={ner_eval['precision']:.2f}  R={ner_eval['recall']:.2f}  F1={ner_eval['f1']:.2f}")
    if ner_eval["fn"]:
        print(f"    Missed: {ner_eval['fn']}")
    if ner_eval["fp"]:
        print(f"    Extra:  {ner_eval['fp']}")

## 7. Verdict

**Expected conclusion:** For a controlled-vocabulary domain like Limbus Company, **rule-based extraction** (PhraseMatcher + regex) achieves high precision and recall without requiring annotated training data. spaCy's EntityRuler provides similar accuracy with cleaner entity labeling (typed entities), making it the recommended approach for the pipeline.

A fully trained statistical NER model would only be worth the effort if the vocabulary grows unpredictably or new mechanics can't be anticipated — unlikely for a game-wiki domain with structured pages.